# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # suppress pandas SettingWithCopyWarning

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect all record sets and their constituent fields, with their `@id`s, as provided by the Croissant schema.

In [ ]:
# List all available record sets and their fields using @id references
if hasattr(metadata, 'record_sets'):
    print("Available record sets (by @id and name):")
    for rs in metadata.record_sets:
        print(f"  RecordSet @id: {rs.id}, name: {getattr(rs, 'name', '[No name]')}")
        if hasattr(rs, 'fields'):
            print("    Fields (by @id and name):")
            for field in rs.fields:
                print(f"      Field @id: {field.id}, name: {getattr(field, 'name', '[No name]')}, type: {getattr(field, 'data_type', '[No type]')}")
else:
    print("No record sets declared in metadata.")

# If possible, retrieve the first available record set @id for demo
main_record_set_id = None
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    main_record_set_id = metadata.record_sets[0].id
    print(f"\nSample records from main record set (@id: {main_record_set_id}):")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break  # Show just a couple of records

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview step.

Below, we extract the main table as a DataFrame.

In [ ]:
# Collect all record set @ids present
record_set_ids = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id for rs in metadata.record_sets]
    print("Loaded record_set @ids:", record_set_ids)
else:
    print("No record sets found. Unable to proceed with extraction.")

# Extract all record sets to DataFrames
dataframes = dict()
for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

# Preview columns of main record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Columns in record set '{main_rs}':\n", dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Let's:
- Identify a numeric field (such as patient age or diagnosis interval by the `@id` from metadata).
- Filter and normalize.
- Perform groupby analysis by a categorical field (such as sex or cancer subtype).

In [ ]:
# EDA Example
import numpy as np

# For demonstration, try to auto-select a likely numeric field (e.g., containing 'age', 'interval', 'years', 'months')
main_df = dataframes.get(main_rs)
numeric_field_id = None
for col in main_df.columns:
    if any(x in col.lower() for x in ['age', 'interval', 'duration', 'months', 'years']):
        numeric_field_id = col
        break

if not numeric_field_id:
    print("No numeric field matching 'age', 'interval', etc. found. Selecting numeric dtype column instead.")
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

print(f"Selected numeric field for analysis: {numeric_field_id}")
# Try also to pick a categorical/group field (e.g. containing 'sex', 'type', 'status', 'location', etc.)
group_field_id = None
for col in main_df.columns:
    if any(x in col.lower() for x in ['sex', 'gender', 'type', 'status', 'site', 'location', 'histology', 'subtype']):
        group_field_id = col
        if col != numeric_field_id:
            break

print(f"Selected group field for analysis: {group_field_id}")

# Apply filter and normalization on the numeric field (e.g., filter > threshold)
if numeric_field_id and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    threshold = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
    print(f"First 5 normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} after filtering, grouped by {group_field_id}:")
        print(grouped_df)
else:
    print("Could not perform EDA: No suitable numeric field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot a histogram of the selected numeric field and, if possible, a boxplot grouped by the categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')

if numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(14,5))
    # Histogram
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15, ax=ax[0])
    ax[0].set_title(f'Distribution of {numeric_field_id}')
    ax[0].set_xlabel(numeric_field_id)

    # Boxplot by group
    if group_field_id and group_field_id in main_df.columns:
        sns.boxplot(y=numeric_field_id, x=group_field_id, data=main_df, ax=ax[1])
        ax[1].set_title(f'{numeric_field_id} by {group_field_id}')
        ax[1].set_xlabel(group_field_id)
        ax[1].set_ylabel(numeric_field_id)
    else:
        ax[1].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field identified for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore a tabular clinical and molecular oncology dataset described by a Croissant schema. We:
- Loaded metadata and records using `mlcroissant`
- Identified available record sets and fields, referencing by their `@id`s
- Loaded the primary data table and performed exploratory analysis
- Normalized a numerical variable and grouped results by a categorical attribute
- Visualized distributions and group differences

For further work, refine the analysis and visualizations using full clinical definitions and documentation, and apply more advanced statistical or machine learning techniques as appropriate.